In [49]:
import pandas as pd

In [50]:
emissions = pd.read_csv("/Users/alexa/Projects/ssp_libya/ssp_modeling/tableau/data/decomposed_emissions_libya_2023.csv")

In [51]:
emissions["strategy"].unique()

array(['Strategy TX:BASE', 'BAU', 'Unconditional', 'Conditional',
       'Historical'], dtype=object)

In [52]:
emissions["subsector"].unique()

array(['1.A.1 - Energy Industries',
       '1.A.2 - Manufacturing Industries and Construction',
       '1.A.3 - Transport', '1.A.4 - Other Sectors',
       '1.B - Fugitive emissions from fuels', '2.A - Mineral Industry',
       '2.B - Chemical Industry', '2.C - Metal Industry',
       '2.D - Non-Energy Products from Fuels and Solvent Use',
       '2.F - Product Uses as Substitutes for Ozone Depleting Substances',
       '3.A - Livestock', '3.B - Land',
       '3.C - Aggregate sources and non-CO2 emissions sources on land',
       '4.A - Solid Waste Disposal',
       '4.D - Wastewater Treatment and Discharge', '5- CCSQ'],
      dtype=object)

In [55]:
# Referencia año a año: Strategy TX:BASE (no el valor fijo de 2025 en Conditional).
# Reducción acumulada 2026–2035 = suma de (BASE − Conditional) por año.
sub = "3.B - Land"
strat_policy = "Conditional"
strat_base = "BAU"

subset = emissions[
    (emissions["subsector"] == sub)
    & (emissions["strategy"].isin([strat_base, strat_policy]))
]
wide = subset.pivot_table(
    index="Year",
    columns="strategy",
    values="value",
    aggfunc="sum",
)
mask_years = wide.index.to_series().between(2026, 2050, inclusive="both")
reduccion_acum_vs_base = (
    wide.loc[mask_years, strat_base] - wide.loc[mask_years, strat_policy]
).sum()
reduccion_acum_vs_base

np.float64(4.929200287206616)

In [56]:
sub = "3.B - Land"
strat_policy = "Conditional"
strat_base = "BAU"
y0, y1 = 2026, 2050

subset = emissions[
    (emissions["subsector"] == sub)
    & (emissions["strategy"].isin([strat_base, strat_policy]))
]
wide = subset.pivot_table(
    index="Year",
    columns="strategy",
    values="value",
    aggfunc="sum",
).sort_index()

mask = wide.index.to_series().between(y0, y1, inclusive="both")

# Reducción neta de emisiones (Gg CO2e acumulado en el periodo): BAU − Policy
# Positivo ⇒ Policy tiene menor emisión neta acumulada que BAU (mitigación adicional vs BAU)
delta_net_emissions = wide.loc[mask, strat_base] - wide.loc[mask, strat_policy]
reduccion_acum_vs_base = delta_net_emissions.sum()

# Comprobar convención: en años sueltos, si Policy es "mejor", ¿BAU − Policy > 0?
# (útil para 3.B con valores negativos)
wide.loc[mask, [strat_base, strat_policy]].head(), delta_net_emissions.head()

(strategy       BAU  Conditional
 Year                           
 2026     -0.751179    -0.751179
 2027     -0.778464    -0.778464
 2028     -0.805694    -0.805694
 2029     -0.832809    -0.875813
 2030     -0.859762    -0.918430,
 Year
 2026    0.000000
 2027    0.000000
 2028    0.000000
 2029    0.043004
 2030    0.058668
 dtype: float64)

In [57]:
reduccion_acum_vs_base

np.float64(4.929200287206616)